# 02 — Customer Analysis
RFM segmentation, LTV, cohort retention, acquisition channel analysis.

In [ ]:

import sys; sys.path.insert(0, '..')
from pipeline.load import get_connection
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid')

con = get_connection()
c360 = con.execute("SELECT * FROM mart_customer_360").df()
customers = con.execute("SELECT * FROM customers").df()


## RFM Segment Distribution

In [ ]:

seg = c360[c360['rfm_segment'].notna()]['rfm_segment'].value_counts()
colors = {'Champions':'#2ecc71','Loyal':'#3498db','Potential Loyalist':'#f1c40f','At Risk':'#e67e22','Lost':'#e74c3c'}
fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(seg.index, seg.values, color=[colors.get(s, 'gray') for s in seg.index], edgecolor='white')
ax.bar_label(bars, fmt='%d', padding=3)
ax.set_title('RFM Segment Distribution'); ax.set_ylabel('Customers')
plt.xticks(rotation=20); plt.tight_layout(); plt.show()
print(seg.to_frame('count').assign(pct=(seg/seg.sum()*100).round(1)))


## Average LTV by RFM Segment

In [ ]:

ltv = c360[c360['rfm_segment'].notna()].groupby('rfm_segment')['total_revenue'].mean().sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(8, 4))
ltv.plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Average LTV by RFM Segment'); ax.set_ylabel('Avg Revenue ($)')
plt.xticks(rotation=20); plt.tight_layout(); plt.show()


## Loyalty Tier vs Revenue

In [ ]:

tier_rev = c360[c360['has_purchased']].groupby('loyalty_tier')[['total_revenue','total_orders','avg_order_value']].mean().round(2)
print(tier_rev)
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, col, color in zip(axes, tier_rev.columns, ['steelblue','coral','mediumpurple']):
    tier_rev[col].plot(kind='bar', ax=ax, color=color, edgecolor='white')
    ax.set_title(col); ax.set_xlabel('Tier')
    plt.sca(ax); plt.xticks(rotation=0)
plt.suptitle('Loyalty Tier Performance'); plt.tight_layout(); plt.show()


## Acquisition Channel Effectiveness

In [ ]:

acq = c360[c360['has_purchased']].groupby('acquisition_channel').agg(
    customers=('customer_id', 'count'),
    avg_ltv=('total_revenue', 'mean'),
    avg_orders=('total_orders', 'mean')
).round(2).sort_values('avg_ltv', ascending=False)
print(acq)
fig, ax = plt.subplots(figsize=(9, 4))
acq['avg_ltv'].plot(kind='bar', ax=ax, color='teal', edgecolor='white')
ax.set_title('Avg LTV by Acquisition Channel'); ax.set_ylabel('Avg Revenue ($)')
plt.xticks(rotation=30, ha='right'); plt.tight_layout(); plt.show()


## Customer Signup Trend

In [ ]:

customers['signup_month'] = pd.to_datetime(customers['signup_date']).dt.to_period('M').astype(str)
monthly_signups = customers.groupby(['signup_month','acquisition_channel'])['customer_id'].count().unstack(fill_value=0)
fig, ax = plt.subplots(figsize=(13, 5))
monthly_signups.plot(kind='area', ax=ax, alpha=0.7, stacked=True)
ax.set_title('Monthly Customer Signups by Acquisition Channel')
ax.set_xlabel('Month'); ax.set_ylabel('New Customers')
plt.xticks(rotation=45, ha='right'); plt.tight_layout(); plt.show()


## Country Revenue Map (Top 10)

In [ ]:

country_rev = c360[c360['has_purchased']].groupby('country')['total_revenue'].sum().sort_values(ascending=False).head(10)
fig, ax = plt.subplots(figsize=(9, 4))
country_rev.plot(kind='bar', ax=ax, color='coral', edgecolor='white')
ax.set_title('Top 10 Countries by Total Revenue'); ax.set_ylabel('Revenue ($)')
plt.xticks(rotation=0); plt.tight_layout(); plt.show()
